# IEEE-CIS Fraud Detection with XGBoost

This notebook develops a fraud-classification model using the IEEE-CIS dataset. It starts with a selected-V-feature baseline, expands to the full V-feature set, compares feature-selection strategies, and selects a decision threshold using validation data.

**Evaluation approach:** data is split chronologically into training (70%), validation (15%), and test (15%) periods. Validation data is used for model selection and threshold tuning. The test set was also examined in earlier experiments, so its results should be interpreted with that limitation in mind.

In [ ]:
!pip install kagglehub

In [ ]:
!pip install xgboost

In [ ]:
import kagglehub
# You may need to re-run this cell after logging in.
kagglehub.login()
path = kagglehub.competition_download('ieee-fraud-detection')

In [ ]:
import os

print(path)
print(os.listdir(path))

In [ ]:
import os
import gc

import pandas as pd
import numpy as np

from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier

In [ ]:
import xgboost as xgb

print("XGBoost version:", xgb.__version__)

XGBoost version: 3.4.1


## 1. Load the dataset

Load the transaction and identity tables from the Kaggle competition files.

In [ ]:
train_transaction = pd.read_csv(
    os.path.join(path, "train_transaction.csv")
)

train_identity = pd.read_csv(
    os.path.join(path, "train_identity.csv")
)

print(train_transaction.shape)
print(train_identity.shape)

(590540, 394)
(144233, 41)


## 2. Define feature groups and initial feature set

Identify C, D, and selected V features, along with categorical features used in the initial XGBoost experiment.

In [ ]:
c_cols = [
    col for col in train_transaction.columns
    if col.startswith("C") and col[1:].isdigit()
]

d_cols = [
    col for col in train_transaction.columns
    if col.startswith("D") and col[1:].isdigit()
]

selected_v_features = [
    "V257", "V244", "V201", "V233", "V188",
    "V156", "V45", "V258", "V171", "V228"
]

print("C features:", len(c_cols))
print("D features:", len(d_cols))
print("Selected V:", len(selected_v_features))

C features: 14
D features: 15
Selected V: 10


In [ ]:
onehot_features = [
    "ProductCD", "card3", "card4", "card5", "card6",
    "addr1", "addr2", "P_emaildomain", "R_emaildomain",
    "M1", "M2", "M3", "M4", "M5",
    "M6", "M7", "M8", "M9"
]

In [ ]:
feature_columns = (
    ["TransactionDT", "TransactionAmt"]
    + ["card1", "card2"]
    + onehot_features
    + c_cols
    + d_cols
    + selected_v_features
)

feature_columns = list(dict.fromkeys(feature_columns))

print("Total features:", len(feature_columns))
print(feature_columns)

Total features: 61
['TransactionDT', 'TransactionAmt', 'card1', 'card2', 'ProductCD', 'card3', 'card4', 'card5', 'card6', 'addr1', 'addr2', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14', 'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9', 'D10', 'D11', 'D12', 'D13', 'D14', 'D15', 'V257', 'V244', 'V201', 'V233', 'V188', 'V156', 'V45', 'V258', 'V171', 'V228']


In [ ]:
model_data = train_transaction[feature_columns + ["isFraud"]].copy()

print("Model data shape:", model_data.shape)
print("Memory usage (GB):", round(model_data.memory_usage(deep=True).sum() / 1024**3, 2))

Model data shape: (590540, 62)
Memory usage (GB): 0.55


## 3. Create a chronological train/validation/test split

Sort transactions by `TransactionDT` before splitting so validation and test periods occur after the training period.

In [ ]:
model_data = model_data.sort_values("TransactionDT").reset_index(drop=True)

n = len(model_data)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_data = model_data.iloc[:train_end]
val_data = model_data.iloc[train_end:val_end]
test_data = model_data.iloc[val_end:]

print("Train:", train_data.shape)
print("Validation:", val_data.shape)
print("Test:", test_data.shape)

print("\nTransactionDT ranges:")
print("Train:", train_data["TransactionDT"].min(), "→", train_data["TransactionDT"].max())
print("Validation:", val_data["TransactionDT"].min(), "→", val_data["TransactionDT"].max())
print("Test:", test_data["TransactionDT"].min(), "→", test_data["TransactionDT"].max())

Train: (413378, 62)
Validation: (88581, 62)
Test: (88581, 62)

TransactionDT ranges:
Train: 86400 → 10437996
Validation: 10438003 → 13151840
Test: 13151880 → 15811131


In [ ]:
X_train = train_data.drop(columns=["isFraud", "TransactionDT"])
y_train = train_data["isFraud"]

X_val = val_data.drop(columns=["isFraud", "TransactionDT"])
y_val = val_data["isFraud"]

X_test = test_data.drop(columns=["isFraud", "TransactionDT"])
y_test = test_data["isFraud"]

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

print("\nFraud rates:")
print("Train:", y_train.mean())
print("Validation:", y_val.mean())
print("Test:", y_test.mean())

X_train: (413378, 60)
X_val: (88581, 60)
X_test: (88581, 60)

Fraud rates:
Train: 0.03516878014795175
Validation: 0.03434145019812375
Test: 0.03480430340592226


## 4. Preprocess the initial feature set

Frequency-encode card features using training-set frequencies, one-hot encode categorical features, and assemble sparse matrices.

In [ ]:
frequency_features = ["card1", "card2"]

for col in frequency_features:
    freq_map = X_train[col].value_counts()

    X_train[col] = X_train[col].map(freq_map)
    X_val[col] = X_val[col].map(freq_map)
    X_test[col] = X_test[col].map(freq_map)

    # Any category that appears only in validation/test gets 0
    X_val[col] = X_val[col].fillna(0)
    X_test[col] = X_test[col].fillna(0)

print(X_train[frequency_features].head())
print("\nMissing values after frequency encoding:")
print(X_train[frequency_features].isna().sum())
print(X_val[frequency_features].isna().sum())
print(X_test[frequency_features].isna().sum())

   card1    card2
0     28      NaN
1    522   2356.0
2    769  26051.0
3   2944   4285.0
4      9  10494.0

Missing values after frequency encoding:
card1       0
card2    6520
dtype: int64
card1    0
card2    0
dtype: int64
card1    0
card2    0
dtype: int64


In [ ]:
from sklearn.preprocessing import OneHotEncoder

categorical_features = onehot_features

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)

X_train_cat = encoder.fit_transform(X_train[categorical_features])
X_val_cat = encoder.transform(X_val[categorical_features])
X_test_cat = encoder.transform(X_test[categorical_features])

print("Encoded categorical shape:")
print("Train:", X_train_cat.shape)
print("Validation:", X_val_cat.shape)
print("Test:", X_test_cat.shape)

Encoded categorical shape:
Train: (413378, 768)
Validation: (88581, 768)
Test: (88581, 768)


In [ ]:
numerical_features = (
    ["TransactionAmt"]
    + c_cols
    + d_cols
    + selected_v_features
)

print("Numerical features:", len(numerical_features))

Numerical features: 40


In [ ]:
from scipy.sparse import csr_matrix, hstack

numeric_features = ["card1", "card2"] + numerical_features

X_train_num = csr_matrix(
    X_train[numeric_features].to_numpy(dtype=np.float32)
)

X_val_num = csr_matrix(
    X_val[numeric_features].to_numpy(dtype=np.float32)
)

X_test_num = csr_matrix(
    X_test[numeric_features].to_numpy(dtype=np.float32)
)

X_train_final = hstack([X_train_num, X_train_cat], format="csr")
X_val_final = hstack([X_val_num, X_val_cat], format="csr")
X_test_final = hstack([X_test_num, X_test_cat], format="csr")

print("Final shapes:")
print("Train:", X_train_final.shape)
print("Validation:", X_val_final.shape)
print("Test:", X_test_final.shape)

Final shapes:
Train: (413378, 810)
Validation: (88581, 810)
Test: (88581, 810)


In [ ]:
negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()

scale_pos_weight = negative_count / positive_count

print("Legitimate:", negative_count)
print("Fraud:", positive_count)
print("scale_pos_weight:", scale_pos_weight)

Legitimate: 398840
Fraud: 14538
scale_pos_weight: 27.434310083918007


## 5. Initial XGBoost model using selected V features

Train and evaluate the first model to establish a stronger nonlinear baseline.

In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    objective="binary:logistic",
    eval_metric=["auc", "aucpr"],
    tree_method="hist",
    early_stopping_rounds=50,
    random_state=42,
    n_jobs=2
)

xgb_model.fit(
    X_train_final,
    y_train,
    eval_set=[(X_val_final, y_val)],
    verbose=50
)

[0]	validation_0-auc:0.81381	validation_0-aucpr:0.30937
[50]	validation_0-auc:0.87361	validation_0-aucpr:0.44713
[100]	validation_0-auc:0.89115	validation_0-aucpr:0.47520
[150]	validation_0-auc:0.89740	validation_0-aucpr:0.48225
[200]	validation_0-auc:0.90163	validation_0-aucpr:0.49041
[250]	validation_0-auc:0.90489	validation_0-aucpr:0.49791
[300]	validation_0-auc:0.90817	validation_0-aucpr:0.50418
[350]	validation_0-auc:0.91011	validation_0-aucpr:0.50987
[400]	validation_0-auc:0.91160	validation_0-aucpr:0.51260
[450]	validation_0-auc:0.91347	validation_0-aucpr:0.51781
[500]	validation_0-auc:0.91390	validation_0-aucpr:0.52030
[550]	validation_0-auc:0.91519	validation_0-aucpr:0.52510
[600]	validation_0-auc:0.91602	validation_0-aucpr:0.53052
[650]	validation_0-auc:0.91630	validation_0-aucpr:0.53372
[700]	validation_0-auc:0.91697	validation_0-aucpr:0.53720
[750]	validation_0-auc:0.91746	validation_0-aucpr:0.54070
[800]	validation_0-auc:0.91777	validation_0-aucpr:0.54341
[850]	validation_

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=50,
              enable_categorical=True, eval_metric=['auc', 'aucpr'],
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=5, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=1000, n_jobs=2,
              num_parallel_tree=None, ...)

In [ ]:
y_val_proba = xgb_model.predict_proba(X_val_final)[:, 1]

print("First 10 predicted probabilities:")
print(y_val_proba[:10])

print("\nProbability range:")
print("Minimum:", y_val_proba.min())
print("Maximum:", y_val_proba.max())

First 10 predicted probabilities:
[0.12568577 0.943152   0.50745004 0.06247358 0.00789804 0.10250694
 0.46684265 0.40180892 0.2845425  0.13690466]

Probability range:
Minimum: 7.259185e-05
Maximum: 0.9998789


In [ ]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)

val_roc_auc = roc_auc_score(y_val, y_val_proba)
val_pr_auc = average_precision_score(y_val, y_val_proba)

print("Validation ROC-AUC:", val_roc_auc)
print("Validation PR-AUC:", val_pr_auc)

Validation ROC-AUC: 0.917794336272817
Validation PR-AUC: 0.547604694901892


In [ ]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

thresholds = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]

results = []

for threshold in thresholds:
    y_val_pred = (y_val_proba >= threshold).astype(int)

    precision = precision_score(y_val, y_val_pred, zero_division=0)
    recall = recall_score(y_val, y_val_pred, zero_division=0)
    f1 = f1_score(y_val, y_val_pred, zero_division=0)

    results.append({
        "Threshold": threshold,
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    })

threshold_results = pd.DataFrame(results)

print(threshold_results.to_string(index=False))

 Threshold  Precision   Recall       F1
       0.1   0.073819 0.956607 0.137060
       0.2   0.117655 0.893162 0.207920
       0.3   0.165635 0.818212 0.275499
       0.4   0.222724 0.746220 0.343056
       0.5   0.287825 0.672255 0.403075
       0.6   0.374244 0.610454 0.464018
       0.7   0.482800 0.535174 0.507640
       0.8   0.623709 0.456607 0.527235
       0.9   0.774377 0.357659 0.489319


In [ ]:
from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(
    y_val,
    y_val_proba
)

f1_scores = (
    2 * precision[:-1] * recall[:-1]
    / (precision[:-1] + recall[:-1] + 1e-12)
)

best_idx = np.argmax(f1_scores)

best_threshold = thresholds[best_idx]
best_precision = precision[best_idx]
best_recall = recall[best_idx]
best_f1 = f1_scores[best_idx]

print("Best threshold:", best_threshold)
print("Precision:", best_precision)
print("Recall:", best_recall)
print("F1:", best_f1)

Best threshold: 0.78251487
Precision: 0.5990878938640133
Recall: 0.4750164365548981
F1: 0.5298863219650365


In [ ]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# 1. Predict fraud probabilities on the untouched test set
y_test_proba = xgb_model.predict_proba(X_test_final)[:, 1]

# 2. Ranking metrics
test_roc_auc = roc_auc_score(y_test, y_test_proba)
test_pr_auc = average_precision_score(y_test, y_test_proba)

# 3. Apply the threshold chosen using validation data
y_test_pred = (y_test_proba >= best_threshold).astype(int)

# 4. Classification metrics
test_precision = precision_score(y_test, y_test_pred, zero_division=0)
test_recall = recall_score(y_test, y_test_pred, zero_division=0)
test_f1 = f1_score(y_test, y_test_pred, zero_division=0)

print("TEST RESULTS")
print("-" * 40)
print(f"ROC-AUC:  {test_roc_auc:.4f}")
print(f"PR-AUC:   {test_pr_auc:.4f}")
print(f"Threshold used: {best_threshold:.4f}")
print(f"Precision: {test_precision:.4f}")
print(f"Recall:    {test_recall:.4f}")
print(f"F1-score:  {test_f1:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_test_pred,
    target_names=["Legitimate", "Fraud"],
    zero_division=0
))

TEST RESULTS
----------------------------------------
ROC-AUC:  0.8917
PR-AUC:   0.4922
Threshold used: 0.7825
Precision: 0.5070
Recall:    0.4437
F1-score:  0.4733

Confusion Matrix:
[[84168  1330]
 [ 1715  1368]]

Classification Report:
              precision    recall  f1-score   support

  Legitimate       0.98      0.98      0.98     85498
       Fraud       0.51      0.44      0.47      3083

    accuracy                           0.97     88581
   macro avg       0.74      0.71      0.73     88581
weighted avg       0.96      0.97      0.96     88581



## 6. Expand to all V features

Construct a second feature set containing all available V features and prepare a chronological split for the full-feature experiment.

In [ ]:
v_cols = [
    col for col in train_transaction.columns
    if col.startswith("V") and col[1:].isdigit()
]

print("Total V features:", len(v_cols))
print(v_cols[:10])
print(v_cols[-10:])

Total V features: 339
['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10']
['V330', 'V331', 'V332', 'V333', 'V334', 'V335', 'V336', 'V337', 'V338', 'V339']


In [ ]:
full_feature_columns = (
    ["TransactionDT", "TransactionAmt", "card1", "card2"]
    + onehot_features
    + c_cols
    + d_cols
    + v_cols
)

full_feature_columns = list(dict.fromkeys(full_feature_columns))

print("Total raw columns:", len(full_feature_columns))
print("Expected:", 389)

Total raw columns: 390
Expected: 389


In [ ]:
model_data_full = train_transaction[
    full_feature_columns + ["isFraud"]
].copy()

print("Full model data shape:", model_data_full.shape)
print(
    "Memory usage (GB):",
    round(
        model_data_full.memory_usage(deep=True).sum() / 1024**3,
        2
    )
)

Full model data shape: (590540, 391)
Memory usage (GB): 2.0


In [ ]:
model_data_full.sort_values(
    "TransactionDT",
    inplace=True
)
model_data_full.reset_index(drop=True, inplace=True)

n = len(model_data_full)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

print("Total rows:", n)
print("Train rows:", train_end)
print("Validation rows:", val_end - train_end)
print("Test rows:", n - val_end)

print("\nTime boundaries:")
print(
    "Train:",
    model_data_full["TransactionDT"].iloc[0],
    "→",
    model_data_full["TransactionDT"].iloc[train_end - 1]
)
print(
    "Validation:",
    model_data_full["TransactionDT"].iloc[train_end],
    "→",
    model_data_full["TransactionDT"].iloc[val_end - 1]
)
print(
    "Test:",
    model_data_full["TransactionDT"].iloc[val_end],
    "→",
    model_data_full["TransactionDT"].iloc[-1]
)

Total rows: 590540
Train rows: 413378
Validation rows: 88581
Test rows: 88581

Time boundaries:
Train: 86400 → 10437996
Validation: 10438003 → 13151840
Test: 13151880 → 15811131


In [ ]:
# Split by row positions without copying the full dataframe
train_slice = model_data_full.iloc[:train_end]
val_slice = model_data_full.iloc[train_end:val_end]
test_slice = model_data_full.iloc[val_end:]

# Separate features and target
X_train_full = train_slice.drop(
    columns=["isFraud", "TransactionDT"]
)
y_train_full = train_slice["isFraud"]

X_val_full = val_slice.drop(
    columns=["isFraud", "TransactionDT"]
)
y_val_full = val_slice["isFraud"]

X_test_full = test_slice.drop(
    columns=["isFraud", "TransactionDT"]
)
y_test_full = test_slice["isFraud"]

print("Train:", X_train_full.shape, y_train_full.shape)
print("Validation:", X_val_full.shape, y_val_full.shape)
print("Test:", X_test_full.shape, y_test_full.shape)

Train: (413378, 389) (413378,)
Validation: (88581, 389) (88581,)
Test: (88581, 389) (88581,)


## 7. Preprocess the full feature set

Fit encoders on training data and transform validation/test data with the same fitted preprocessing objects.

In [ ]:
frequency_features = ["card1", "card2"]

for col in frequency_features:
    freq_map = X_train_full[col].value_counts()

    X_train_full[col] = X_train_full[col].map(freq_map)
    X_val_full[col] = X_val_full[col].map(freq_map)
    X_test_full[col] = X_test_full[col].map(freq_map)

    # Preserve missing values; XGBoost handles them natively
    # Unseen categories also become NaN and are handled by XGBoost

print("Train missing values:")
print(X_train_full[frequency_features].isna().sum())

print("\nValidation missing values:")
print(X_val_full[frequency_features].isna().sum())

print("\nTest missing values:")
print(X_test_full[frequency_features].isna().sum())

Train missing values:
card1       0
card2    6520
dtype: int64

Validation missing values:
card1    967
card2    800
dtype: int64

Test missing values:
card1    1210
card2    1613
dtype: int64


In [ ]:
from sklearn.preprocessing import OneHotEncoder

categorical_features = onehot_features

encoder_full = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)

X_train_cat_full = encoder_full.fit_transform(
    X_train_full[categorical_features]
)

X_val_cat_full = encoder_full.transform(
    X_val_full[categorical_features]
)

X_test_cat_full = encoder_full.transform(
    X_test_full[categorical_features]
)

print("Encoded categorical shape:")
print("Train:", X_train_cat_full.shape)
print("Validation:", X_val_cat_full.shape)
print("Test:", X_test_cat_full.shape)

Encoded categorical shape:
Train: (413378, 768)
Validation: (88581, 768)
Test: (88581, 768)


In [ ]:
from scipy.sparse import csr_matrix, hstack

numerical_features_full = (
    ["card1", "card2", "TransactionAmt"]
    + c_cols
    + d_cols
    + v_cols
)

print("Numerical features:", len(numerical_features_full))

X_train_num_full = csr_matrix(
    X_train_full[numerical_features_full].to_numpy(
        dtype=np.float32
    )
)

X_val_num_full = csr_matrix(
    X_val_full[numerical_features_full].to_numpy(
        dtype=np.float32
    )
)

X_test_num_full = csr_matrix(
    X_test_full[numerical_features_full].to_numpy(
        dtype=np.float32
    )
)

X_train_final_full = hstack(
    [X_train_num_full, X_train_cat_full],
    format="csr"
)

X_val_final_full = hstack(
    [X_val_num_full, X_val_cat_full],
    format="csr"
)

X_test_final_full = hstack(
    [X_test_num_full, X_test_cat_full],
    format="csr"
)

print("\nFinal shapes:")
print("Train:", X_train_final_full.shape)
print("Validation:", X_val_final_full.shape)
print("Test:", X_test_final_full.shape)

Numerical features: 371

Final shapes:
Train: (413378, 1139)
Validation: (88581, 1139)
Test: (88581, 1139)


## 8. Train the full-V XGBoost model

Train XGBoost using all V features and the same core hyperparameters for comparison.

In [ ]:
from xgboost import XGBClassifier

xgb_model_full = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    objective="binary:logistic",
    eval_metric=["auc", "aucpr"],
    tree_method="hist",
    early_stopping_rounds=50,
    random_state=42,
    n_jobs=2
)

xgb_model_full.fit(
    X_train_final_full,
    y_train_full,
    eval_set=[(X_val_final_full, y_val_full)],
    verbose=50
)

[0]	validation_0-auc:0.82782	validation_0-aucpr:0.32795
[50]	validation_0-auc:0.87929	validation_0-aucpr:0.46255
[100]	validation_0-auc:0.89447	validation_0-aucpr:0.48889
[150]	validation_0-auc:0.90057	validation_0-aucpr:0.50089
[200]	validation_0-auc:0.90518	validation_0-aucpr:0.50966
[250]	validation_0-auc:0.90740	validation_0-aucpr:0.51347
[300]	validation_0-auc:0.91051	validation_0-aucpr:0.51945
[350]	validation_0-auc:0.91211	validation_0-aucpr:0.52276
[400]	validation_0-auc:0.91339	validation_0-aucpr:0.52683
[450]	validation_0-auc:0.91385	validation_0-aucpr:0.52872
[500]	validation_0-auc:0.91484	validation_0-aucpr:0.53149
[550]	validation_0-auc:0.91636	validation_0-aucpr:0.53635
[600]	validation_0-auc:0.91643	validation_0-aucpr:0.53785
[650]	validation_0-auc:0.91609	validation_0-aucpr:0.53961
[700]	validation_0-auc:0.91646	validation_0-aucpr:0.54258
[750]	validation_0-auc:0.91619	validation_0-aucpr:0.54445
[800]	validation_0-auc:0.91592	validation_0-aucpr:0.54572
[850]	validation_

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=50,
              enable_categorical=True, eval_metric=['auc', 'aucpr'],
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=5, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=1000, n_jobs=2,
              num_parallel_tree=None, ...)

In [ ]:
print("Best iteration:", xgb_model_full.best_iteration)
print("Best score:", xgb_model_full.best_score)

Best iteration: 994
Best score: 0.551310657470848


## 9. Evaluate full-V model on validation data

Use ROC-AUC and average precision (PR-AUC) to assess ranking performance.

In [ ]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve
)
import numpy as np

# Predict fraud probabilities on validation data
val_probs_full = xgb_model_full.predict_proba(
    X_val_final_full
)[:, 1]

# Evaluate ranking performance
val_roc_auc_full = roc_auc_score(
    y_val_full, val_probs_full
)

val_pr_auc_full = average_precision_score(
    y_val_full, val_probs_full
)

print(f"Validation ROC-AUC: {val_roc_auc_full:.4f}")
print(f"Validation PR-AUC:  {val_pr_auc_full:.4f}")

Validation ROC-AUC: 0.9149
Validation PR-AUC:  0.5514


### Full-V model threshold experiment

This earlier threshold calculation belongs to the full-V model and is retained as part of the model comparison. The final candidate's threshold is selected later using the top-100 early-stopped model.


In [ ]:
precision, recall, thresholds = precision_recall_curve(
    y_val_full, val_probs_full
)

# The last precision/recall point has no corresponding threshold
f1_scores = (
    2 * precision[:-1] * recall[:-1]
    / (precision[:-1] + recall[:-1] + 1e-10)
)

best_idx = np.argmax(f1_scores)
best_threshold_full = thresholds[best_idx]

print(f"Best validation threshold: {best_threshold_full:.6f}")
print(f"Validation precision:      {precision[best_idx]:.4f}")
print(f"Validation recall:         {recall[best_idx]:.4f}")
print(f"Validation F1-score:       {f1_scores[best_idx]:.4f}")

Best validation threshold: 0.785104
Validation precision:      0.6399
Validation recall:         0.4563
Validation F1-score:       0.5327


### Earlier test-set comparison

This cell evaluates the full-V model on the test period as part of the earlier experiments. Because test results were examined during model comparison, the test set is not completely untouched.

In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score
)

# Predict test probabilities
test_probs_full = xgb_model_full.predict_proba(
    X_test_final_full
)[:, 1]

# Threshold selected on validation data
test_preds_full = (
    test_probs_full >= best_threshold_full
).astype(int)

print("Test ROC-AUC:",
      round(roc_auc_score(y_test_full, test_probs_full), 4))

print("Test PR-AUC:",
      round(average_precision_score(y_test_full, test_probs_full), 4))

print("Test Precision:",
      round(precision_score(y_test_full, test_preds_full), 4))

print("Test Recall:",
      round(recall_score(y_test_full, test_preds_full), 4))

print("Test F1-score:",
      round(f1_score(y_test_full, test_preds_full), 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_full, test_preds_full))

print("\nClassification Report:")
print(classification_report(
    y_test_full,
    test_preds_full,
    target_names=["Legitimate", "Fraud"],
    digits=4
))

Test ROC-AUC: 0.8879
Test PR-AUC: 0.5033
Test Precision: 0.5657
Test Recall: 0.433
Test F1-score: 0.4905

Confusion Matrix:
[[84473  1025]
 [ 1748  1335]]

Classification Report:
              precision    recall  f1-score   support

  Legitimate     0.9797    0.9880    0.9839     85498
       Fraud     0.5657    0.4330    0.4905      3083

    accuracy                         0.9687     88581
   macro avg     0.7727    0.7105    0.7372     88581
weighted avg     0.9653    0.9687    0.9667     88581



## 10. Inspect feature importance

Reconstruct the feature-name mapping and inspect overall and V-feature importance scores. Importance indicates model reliance, not causality.

In [ ]:
# Get names of the one-hot encoded columns
encoded_feature_names = encoder.get_feature_names_out(
    onehot_features
)

# Combine names in the same order as the sparse matrix
all_feature_names = (
    list(numerical_features) +
    list(encoded_feature_names)
)

print("Number of feature names:", len(all_feature_names))
print("Number of model features:", X_train_final_full.shape[1])

Number of feature names: 808
Number of model features: 1139


In [ ]:
print("Numerical feature list:", len(numerical_features_full))
print("Numerical matrix columns:", X_train_num_full.shape[1])

print("One-hot matrix columns:", X_train_cat_full.shape[1])
print("Final matrix columns:", X_train_final_full.shape[1])

print("Encoded feature names:",
      len(encoder.get_feature_names_out(onehot_features)))

Numerical feature list: 371
Numerical matrix columns: 371
One-hot matrix columns: 768
Final matrix columns: 1139
Encoded feature names: 768


In [ ]:
encoded_feature_names = encoder.get_feature_names_out(
    onehot_features
)

all_feature_names_full = (
    numerical_features_full
    + list(encoded_feature_names)
)

assert len(all_feature_names_full) == X_train_final_full.shape[1]

print("Feature names reconstructed successfully!")

Feature names reconstructed successfully!


In [ ]:
import pandas as pd

importance_full = pd.Series(
    xgb_model_full.feature_importances_,
    index=all_feature_names_full
).sort_values(ascending=False)

print("Top 20 features overall:")
print(importance_full.head(20))

print("\nTop 30 V features:")
print(
    importance_full[
        importance_full.index.str.match(r"^V\d+$")
    ].head(30)
)

Top 20 features overall:
V258           0.107180
V218           0.035598
V70            0.034879
V90            0.026845
V201           0.026567
V294           0.022324
ProductCD_C    0.016204
addr2_87.0     0.016090
M4_nan         0.012517
V187           0.010875
V264           0.009797
C8             0.009096
V287           0.008159
C4             0.007788
card6_debit    0.006878
C14            0.006770
V34            0.006053
V91            0.005475
V308           0.005461
V283           0.005284
dtype: float32

Top 30 V features:
V258    0.107180
V218    0.035598
V70     0.034879
V90     0.026845
V201    0.026567
V294    0.022324
V187    0.010875
V264    0.009797
V287    0.008159
V34     0.006053
V91     0.005475
V308    0.005461
V283    0.005284
V41     0.005152
V317    0.004150
V45     0.003648
V102    0.003636
V33     0.003157
V151    0.003037
V296    0.002960
V58     0.002776
V312    0.002698
V163    0.002586
V189    0.002434
V49     0.002338
V67     0.002241
V50     0.002234
V

## 11. Save model and preprocessing artifacts

Save the full-V model, encoder, feature names, sparse matrices, and labels to Google Drive for later experiments. These artifacts should not be committed to GitHub.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

save_dir = "/content/drive/MyDrive/transaction-fraud-detection"
os.makedirs(save_dir, exist_ok=True)

In [ ]:
model_path = os.path.join(save_dir, "xgb_model_full.json")

xgb_model_full.save_model(model_path)

print("Model saved:", model_path)

Model saved: /content/drive/MyDrive/transaction-fraud-detection/xgb_model_full.json


In [ ]:
import joblib
import json

joblib.dump(
    encoder,
    os.path.join(save_dir, "encoder.joblib")
)

with open(
    os.path.join(save_dir, "feature_names.json"), "w"
) as f:
    json.dump(all_feature_names_full, f)

with open(
    os.path.join(save_dir, "numerical_features.json"), "w"
) as f:
    json.dump(numerical_features_full, f)

with open(
    os.path.join(save_dir, "v_features.json"), "w"
) as f:
    json.dump(v_cols, f)

print("Preprocessing artifacts saved!")

Preprocessing artifacts saved!


In [ ]:
from scipy.sparse import save_npz
import numpy as np

matrices = {
    "X_train_final_full": X_train_final_full,
    "X_val_final_full": X_val_final_full,
    "X_test_final_full": X_test_final_full,
    "X_train_cat_full": X_train_cat_full,
    "X_val_cat_full": X_val_cat_full,
    "X_test_cat_full": X_test_cat_full,
}

for name, matrix in matrices.items():
    save_npz(
        os.path.join(save_dir, f"{name}.npz"),
        matrix
    )

np.save(
    os.path.join(save_dir, "y_train_full.npy"),
    np.asarray(y_train_full)
)

np.save(
    os.path.join(save_dir, "y_val_full.npy"),
    np.asarray(y_val_full)
)

np.save(
    os.path.join(save_dir, "y_test_full.npy"),
    np.asarray(y_test_full)
)

print("Processed matrices and labels saved!")

Processed matrices and labels saved!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

save_dir = "/content/drive/MyDrive/transaction-fraud-detection"

print("Folder exists:", os.path.exists(save_dir))
print("Saved files:", os.listdir(save_dir))

Folder exists: True
Saved files: ['xgb_model_full.json', 'encoder.joblib', 'feature_names.json', 'numerical_features.json', 'v_features.json', 'X_train_final_full.npz', 'X_val_final_full.npz', 'X_test_final_full.npz', 'X_train_cat_full.npz', 'X_val_cat_full.npz', 'X_test_cat_full.npz', 'y_train_full.npy', 'y_val_full.npy', 'y_test_full.npy']


## 12. Reload saved artifacts and rank V features

Restore the saved model and matrices, then derive the V-feature ranking used in the feature-selection experiments.

In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd

from scipy.sparse import load_npz
from xgboost import XGBClassifier

save_dir = "/content/drive/MyDrive/transaction-fraud-detection"

# Load trained XGBoost model
xgb_model_full = XGBClassifier()
xgb_model_full.load_model(
    os.path.join(save_dir, "xgb_model_full.json")
)

# Load feature names
with open(os.path.join(save_dir, "feature_names.json")) as f:
    all_feature_names_full = json.load(f)

with open(os.path.join(save_dir, "numerical_features.json")) as f:
    numerical_features_full = json.load(f)

with open(os.path.join(save_dir, "v_features.json")) as f:
    v_cols = json.load(f)

# Load encoder
encoder = joblib.load(
    os.path.join(save_dir, "encoder.joblib")
)

# Load processed training and validation matrices
X_train_final_full = load_npz(
    os.path.join(save_dir, "X_train_final_full.npz")
)

X_val_final_full = load_npz(
    os.path.join(save_dir, "X_val_final_full.npz")
)

# Load labels
y_train_full = np.load(
    os.path.join(save_dir, "y_train_full.npy")
)

y_val_full = np.load(
    os.path.join(save_dir, "y_val_full.npy")
)

print("Model and preprocessing artifacts loaded!")
print("Training shape:", X_train_final_full.shape)
print("Validation shape:", X_val_final_full.shape)
print("Feature names:", len(all_feature_names_full))
print("V features:", len(v_cols))

Model and preprocessing artifacts loaded!
Training shape: (413378, 1139)
Validation shape: (88581, 1139)
Feature names: 1139
V features: 339


In [ ]:
assert len(all_feature_names_full) == X_train_final_full.shape[1]

importance_full = pd.Series(
    xgb_model_full.feature_importances_,
    index=all_feature_names_full
).sort_values(ascending=False)

# Extract importance scores for V features only
v_importance = importance_full[
    importance_full.index.str.match(r"^V\d+$")
]

print("Top 25 V features:")
print(v_importance.head(25))

Top 25 V features:
V258    0.107180
V218    0.035598
V70     0.034879
V90     0.026845
V201    0.026567
V294    0.022324
V187    0.010875
V264    0.009797
V287    0.008159
V34     0.006053
V91     0.005475
V308    0.005461
V283    0.005284
V41     0.005152
V317    0.004150
V45     0.003648
V102    0.003636
V33     0.003157
V151    0.003037
V296    0.002960
V58     0.002776
V312    0.002698
V163    0.002586
V189    0.002434
V49     0.002338
dtype: float32


## 13. Compare reduced V-feature sets

Keep all non-V features and retain only the highest-ranked V features. Compare top-25, top-50, and top-100 V-feature configurations using the same training and validation split.

In [ ]:
# Select the top 25 V features from the training model's ranking
top_25_v = v_importance.head(25).index.tolist()

# Keep all non-V features and only the selected V features
selected_features_25 = [
    name for name in all_feature_names_full
    if not name.startswith("V")
    or name in top_25_v
]

# Find their column positions in the original matrix
selected_indices_25 = [
    i for i, name in enumerate(all_feature_names_full)
    if name in selected_features_25
]

print("Selected V features:", len(top_25_v))
print("Total features retained:", len(selected_indices_25))
print("Features removed:", X_train_final_full.shape[1] - len(selected_indices_25))

Selected V features: 25
Total features retained: 825
Features removed: 314


In [ ]:
X_train_25 = X_train_final_full[:, selected_indices_25]
X_val_25 = X_val_final_full[:, selected_indices_25]

print("Training shape:", X_train_25.shape)
print("Validation shape:", X_val_25.shape)

Training shape: (413378, 825)
Validation shape: (88581, 825)


In [ ]:
from xgboost import XGBClassifier

scale_pos_weight = (
    (y_train_full == 0).sum()
    / (y_train_full == 1).sum()
)

xgb_model_25 = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    objective="binary:logistic",
    eval_metric=["auc", "aucpr"],
    tree_method="hist",
    random_state=42,
    n_jobs=2
)

xgb_model_25.fit(
    X_train_25,
    y_train_full,
    eval_set=[(X_val_25, y_val_full)],
    verbose=100
)

[0]	validation_0-auc:0.83411	validation_0-aucpr:0.33945
[100]	validation_0-auc:0.89532	validation_0-aucpr:0.48806
[200]	validation_0-auc:0.90620	validation_0-aucpr:0.50108
[300]	validation_0-auc:0.91326	validation_0-aucpr:0.51409
[400]	validation_0-auc:0.91641	validation_0-aucpr:0.52428
[500]	validation_0-auc:0.91795	validation_0-aucpr:0.53187
[600]	validation_0-auc:0.91960	validation_0-aucpr:0.53752
[700]	validation_0-auc:0.91941	validation_0-aucpr:0.54266
[800]	validation_0-auc:0.91894	validation_0-aucpr:0.54727
[900]	validation_0-auc:0.91930	validation_0-aucpr:0.55136
[999]	validation_0-auc:0.91910	validation_0-aucpr:0.55386


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric=['auc', 'aucpr'],
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=5, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=1000, n_jobs=2,
              num_parallel_tree=None, ...)

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

val_probs_25 = xgb_model_25.predict_proba(X_val_25)[:, 1]

roc_auc_25 = roc_auc_score(y_val_full, val_probs_25)
pr_auc_25 = average_precision_score(y_val_full, val_probs_25)

print(f"Top-25 Validation ROC-AUC: {roc_auc_25:.4f}")
print(f"Top-25 Validation PR-AUC:  {pr_auc_25:.4f}")
print(f"Full-model reference PR-AUC: {0.55131:.5f}")

Top-25 Validation ROC-AUC: 0.9191
Top-25 Validation PR-AUC:  0.5539
Full-model reference PR-AUC: 0.55131


In [ ]:
from scipy.sparse import hstack
from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier

# Select the 50 highest-importance V features
top_50_v = v_importance.head(50).index.tolist()

# Keep all non-V features and only the selected V features
selected_features_50 = [
    name for name in all_feature_names_full
    if not name.startswith("V") or name in top_50_v
]

# Find the original column positions
selected_indices_50 = [
    i for i, name in enumerate(all_feature_names_full)
    if name in selected_features_50
]

# Select columns from existing sparse matrices
X_train_50 = X_train_final_full[:, selected_indices_50]
X_val_50 = X_val_final_full[:, selected_indices_50]

print("Selected V features:", len(top_50_v))
print("Training shape:", X_train_50.shape)
print("Validation shape:", X_val_50.shape)

Selected V features: 50
Training shape: (413378, 850)
Validation shape: (88581, 850)


In [ ]:
xgb_model_50 = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    objective="binary:logistic",
    eval_metric=["auc", "aucpr"],
    tree_method="hist",
    random_state=42,
    n_jobs=2
)

xgb_model_50.fit(
    X_train_50,
    y_train_full,
    eval_set=[(X_val_50, y_val_full)],
    verbose=100
)

[0]	validation_0-auc:0.82458	validation_0-aucpr:0.33008
[100]	validation_0-auc:0.89438	validation_0-aucpr:0.48217
[200]	validation_0-auc:0.90612	validation_0-aucpr:0.49948
[300]	validation_0-auc:0.91189	validation_0-aucpr:0.51399
[400]	validation_0-auc:0.91597	validation_0-aucpr:0.52502
[500]	validation_0-auc:0.91887	validation_0-aucpr:0.53210
[600]	validation_0-auc:0.92037	validation_0-aucpr:0.53837
[700]	validation_0-auc:0.92053	validation_0-aucpr:0.54407
[800]	validation_0-auc:0.92073	validation_0-aucpr:0.54848
[900]	validation_0-auc:0.92046	validation_0-aucpr:0.55175
[999]	validation_0-auc:0.92030	validation_0-aucpr:0.55508


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric=['auc', 'aucpr'],
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=5, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=1000, n_jobs=2,
              num_parallel_tree=None, ...)

In [ ]:
val_probs_50 = xgb_model_50.predict_proba(X_val_50)[:, 1]

roc_auc_50 = roc_auc_score(y_val_full, val_probs_50)
pr_auc_50 = average_precision_score(y_val_full, val_probs_50)

print(f"Top-50 Validation ROC-AUC: {roc_auc_50:.4f}")
print(f"Top-50 Validation PR-AUC:  {pr_auc_50:.4f}")

Top-50 Validation ROC-AUC: 0.9203
Top-50 Validation PR-AUC:  0.5551


In [ ]:
# Select the top 100 V features
top_100_v = v_importance.head(100).index.tolist()

# Keep all non-V features and only the selected V features
selected_features_100 = [
    name for name in all_feature_names_full
    if not name.startswith("V") or name in top_100_v
]

# Find the corresponding column positions
selected_indices_100 = [
    i for i, name in enumerate(all_feature_names_full)
    if name in selected_features_100
]

# Create sparse matrices
X_train_100 = X_train_final_full[:, selected_indices_100]
X_val_100 = X_val_final_full[:, selected_indices_100]

print("Selected V features:", len(top_100_v))
print("Training shape:", X_train_100.shape)
print("Validation shape:", X_val_100.shape)

Selected V features: 100
Training shape: (413378, 900)
Validation shape: (88581, 900)


In [ ]:
xgb_model_100 = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    objective="binary:logistic",
    eval_metric=["auc", "aucpr"],
    tree_method="hist",
    random_state=42,
    n_jobs=2
)

xgb_model_100.fit(
    X_train_100,
    y_train_full,
    eval_set=[(X_val_100, y_val_full)],
    verbose=100
)

[0]	validation_0-auc:0.82414	validation_0-aucpr:0.32537
[100]	validation_0-auc:0.89425	validation_0-aucpr:0.49267
[200]	validation_0-auc:0.90571	validation_0-aucpr:0.51056
[300]	validation_0-auc:0.91228	validation_0-aucpr:0.52203
[400]	validation_0-auc:0.91581	validation_0-aucpr:0.53155
[500]	validation_0-auc:0.91755	validation_0-aucpr:0.53834
[600]	validation_0-auc:0.91851	validation_0-aucpr:0.54487
[700]	validation_0-auc:0.91864	validation_0-aucpr:0.54809
[800]	validation_0-auc:0.91903	validation_0-aucpr:0.55240
[900]	validation_0-auc:0.91947	validation_0-aucpr:0.55598
[999]	validation_0-auc:0.91982	validation_0-aucpr:0.55955


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric=['auc', 'aucpr'],
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=5, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=1000, n_jobs=2,
              num_parallel_tree=None, ...)

In [ ]:
val_probs_100 = xgb_model_100.predict_proba(X_val_100)[:, 1]

roc_auc_100 = roc_auc_score(y_val_full, val_probs_100)
pr_auc_100 = average_precision_score(y_val_full, val_probs_100)

print(f"Top-100 Validation ROC-AUC: {roc_auc_100:.4f}")
print(f"Top-100 Validation PR-AUC:  {pr_auc_100:.4f}")

Top-100 Validation ROC-AUC: 0.9198
Top-100 Validation PR-AUC:  0.5596


In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

val_probs_full = xgb_model_full.predict_proba(
    X_val_final_full
)[:, 1]

print("Full model ROC-AUC:",
      roc_auc_score(y_val_full, val_probs_full))

print("Full model PR-AUC:",
      average_precision_score(y_val_full, val_probs_full))

Full model ROC-AUC: 0.9149168045036057
Full model PR-AUC: 0.5513594494443366


## 14. Train the top-100 model with early stopping

Monitor validation AUC-PR and stop when it no longer improves for 50 rounds.

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

xgb_model_100_es = XGBClassifier(
    n_estimators=1500,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    early_stopping_rounds=50,
    random_state=42,
    n_jobs=2
)

xgb_model_100_es.fit(
    X_train_100,
    y_train_full,
    eval_set=[(X_val_100, y_val_full)],
    verbose=100
)

val_probs_100_es = xgb_model_100_es.predict_proba(
    X_val_100
)[:, 1]

print("Best iteration:", xgb_model_100_es.best_iteration)
print("Validation ROC-AUC:",
      roc_auc_score(y_val_full, val_probs_100_es))
print("Validation PR-AUC:",
      average_precision_score(y_val_full, val_probs_100_es))

[0]	validation_0-aucpr:0.32537
[100]	validation_0-aucpr:0.49267
[200]	validation_0-aucpr:0.51056
[300]	validation_0-aucpr:0.52203
[400]	validation_0-aucpr:0.53155
[500]	validation_0-aucpr:0.53834
[600]	validation_0-aucpr:0.54487
[700]	validation_0-aucpr:0.54809
[800]	validation_0-aucpr:0.55240
[900]	validation_0-aucpr:0.55598
[1000]	validation_0-aucpr:0.55950
[1100]	validation_0-aucpr:0.56190
[1200]	validation_0-aucpr:0.56363
[1296]	validation_0-aucpr:0.56394
Best iteration: 1246
Validation ROC-AUC: 0.9195301366969351
Validation PR-AUC: 0.564596553730337


In [ ]:
import os

save_dir = "/content/drive/MyDrive/transaction-fraud-detection"
os.makedirs(save_dir, exist_ok=True)

xgb_model_100_es.save_model(
    f"{save_dir}/xgb_model_top100_earlystop.json"
)

print("Model saved successfully.")

Model saved successfully.


## 15. Select a decision threshold on validation data

Choose the threshold that maximizes validation F1. This is a model-selection decision, not a threshold optimized on the test set.

In [ ]:

import numpy as np
import pandas as pd
from sklearn.metrics import (
    precision_recall_curve,
    roc_auc_score,
    average_precision_score
)

# Predict fraud probabilities on the validation set
val_probs = xgb_model_100_es.predict_proba(X_val_100)[:, 1]

# Calculate precision and recall at different thresholds
precision, recall, thresholds = precision_recall_curve(
    y_val_full, val_probs
)

# Calculate F1 for each threshold
f1_scores = (
    2 * precision[:-1] * recall[:-1]
    / (precision[:-1] + recall[:-1] + 1e-10)
)

# Find the threshold with maximum F1
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print("Optimal threshold:", best_threshold)
print("Precision:", precision[best_idx])
print("Recall:", recall[best_idx])
print("F1-score:", f1_scores[best_idx])
print("Validation ROC-AUC:",
      roc_auc_score(y_val_full, val_probs))
print("Validation PR-AUC:",
      average_precision_score(y_val_full, val_probs))


Optimal threshold: 0.7830288
Precision: 0.6490036231884058
Recall: 0.4710716633793557
F1-score: 0.5459047618560237
Validation ROC-AUC: 0.9195301366969351
Validation PR-AUC: 0.564596553730337


## 16. Analyze the precision-recall trade-off

Compare alternative thresholds on validation data to illustrate the trade-off between fraud detected and false alarms.

In [ ]:

import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

# Reuse the validation probabilities and labels
val_probs = xgb_model_100_es.predict_proba(X_val_100)[:, 1]

# Thresholds to compare
thresholds_to_check = [
    0.20, 0.30, 0.40, 0.50, 0.60,
    0.70, 0.75, 0.783, 0.80, 0.85, 0.90
]

rows = []

for threshold in thresholds_to_check:
    preds = (val_probs >= threshold).astype(int)

    tp = int(((preds == 1) & (y_val_full == 1)).sum())
    fp = int(((preds == 1) & (y_val_full == 0)).sum())
    fn = int(((preds == 0) & (y_val_full == 1)).sum())

    rows.append({
        "Threshold": threshold,
        "Precision": precision_score(
            y_val_full, preds, zero_division=0
        ),
        "Recall": recall_score(
            y_val_full, preds, zero_division=0
        ),
        "F1": f1_score(
            y_val_full, preds, zero_division=0
        ),
        "Frauds caught": tp,
        "Frauds missed": fn,
        "False alarms": fp,
        "Total flagged": tp + fp
    })

tradeoff_df = pd.DataFrame(rows)
print(tradeoff_df.round(4).to_string(index=False))


 Threshold  Precision  Recall     F1  Frauds caught  Frauds missed  False alarms  Total flagged
     0.200     0.1396  0.8613 0.2402           2620            422         16149          18769
     0.300     0.1990  0.7932 0.3182           2413            629          9711          12124
     0.400     0.2654  0.7117 0.3866           2165            877          5994           8159
     0.500     0.3393  0.6443 0.4445           1960           1082          3816           5776
     0.600     0.4350  0.5884 0.5002           1790           1252          2325           4115
     0.700     0.5428  0.5256 0.5341           1599           1443          1347           2946
     0.750     0.6004  0.4944 0.5423           1504           1538          1001           2505
     0.783     0.6490  0.4711 0.5459           1433           1609           775           2208
     0.800     0.6691  0.4546 0.5414           1383           1659           684           2067
     0.850     0.7278  0.4158 0.5293    

## 17. Final candidate evaluation on the test set

Apply the threshold chosen on validation data without changing it based on test results. Note that the test period was inspected in earlier experiments.

In [ ]:

test_probs = xgb_model_100_es.predict_proba(X_test_100)[:, 1]
test_preds = (test_probs >= best_threshold).astype(int)

from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score,
    confusion_matrix
)

print("Test ROC-AUC:", roc_auc_score(y_test_full, test_probs))
print("Test PR-AUC:", average_precision_score(y_test_full, test_probs))
print("Test Precision:", precision_score(y_test_full, test_preds))
print("Test Recall:", recall_score(y_test_full, test_preds))
print("Test F1:", f1_score(y_test_full, test_preds))
print("Confusion Matrix:\n", confusion_matrix(y_test_full, test_preds))


Test ROC-AUC: 0.8935129389076916
Test PR-AUC: 0.5068095622301365
Test Precision: 0.5869755244755245
Test Recall: 0.43561466104443725
Test F1: 0.5000930925339788
Confusion Matrix:
 [[84553   945]
 [ 1740  1343]]


## Summary

The experiments compare a selected-V-feature XGBoost baseline, a full-V model, reduced V-feature sets, and a top-100 model trained with early stopping. The current leading candidate is the top-100 early-stopped model based on validation average precision. The threshold is selected by maximizing validation F1; threshold choice should be interpreted as a metric-based choice because real-world costs of false positives and false negatives are unavailable.

**Reproducibility note:** The notebook expects the Kaggle dataset and, for the later feature-selection cells, saved artifacts in Google Drive. Review the setup paths before rerunning outside the original Colab environment.